# Linear Algebra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/05_linalg.ipynb)

Runnable examples from **Data/ML Handbook** — section `05_linalg`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/05_linalg/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s04_pca_svd.py`

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
X = load_iris().data; Xc = X - X.mean(0)
evals, evecs = np.linalg.eigh(np.cov(Xc.T)); evals, evecs = evals[::-1], evecs[:, ::-1]
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pca = PCA().fit(X)
print("eig of covariance  :", evals.round(4))
print("S² / (n−1) from SVD:", (S**2 / (len(X) - 1)).round(4))
print("sklearn explained  :", pca.explained_variance_.round(4))
print("explained ratio    :", pca.explained_variance_ratio_.round(3), "-> 2 PCs keep", pca.explained_variance_ratio_[:2].sum().round(3))
print("same directions (|cos|):", np.abs((evecs[:, 0] * Vt[0]).sum()).round(6))

### `code/s04_ols.py`

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = np.c_[np.ones(200), rng.normal(size=(200, 2))]          # bias column + 2 features
w_true = np.array([1.0, 2.0, -3.0]); y = X @ w_true + rng.normal(0, 0.5, 200)
w_normal = np.linalg.solve(X.T @ X, X.T @ y)                  # normal equation (solve, never inv)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)               # QR/SVD based: numerically safer
w = np.zeros(3)                                                # gradient descent on MSE
for _ in range(500): w -= 0.1 * (2 / len(y)) * X.T @ (X @ w - y)
print("normal eq:", w_normal.round(3), "\nlstsq    :", w_lstsq.round(3), "\nGD       :", w.round(3))
print("condition number of XᵀX:", round(np.linalg.cond(X.T @ X), 2))